<a href="https://colab.research.google.com/github/arinamartynyuk/-4-9/blob/main/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%9C%D0%B0%D1%80%D1%82%D0%B8%D0%BD%D1%8E%D0%BA_4_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** __________________________  
**Група:** _________________________________  
**Дата:** __________________________________

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [3]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [4]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [5]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [6]:
# Дослідження набору даних
print('Кількість унікальних користувачів:', ratings_df['user_id'].nunique())
print('Кількість унікальних фільмів:', ratings_df['item_id'].nunique())
print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', ratings_df['rating'].mean())
print('Пропущені значення:\n', ratings_df.isna().sum())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.52986
Пропущені значення:
 user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

Набір MovieLens 100K містить оцінки користувачів різних фільмів. У ньому є багато користувачів і фільмів, але кожен користувач оцінює лише частину каталогу, тому матриця оцінок є розрідженою. Оцінки числові, а їхні статистики наведені вище.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [7]:
# Розбиття даних на навчальну та тестову вибірки (80/20)
train_df, test_df = sklearn_train_test_split(
    ratings_df, test_size=0.2, random_state=42
)
print('Train:', train_df.shape)
print('Test:', test_df.shape)

Train: (80000, 4)
Test: (20000, 4)


In [8]:
# Підготовка даних для Surprise
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']], reader
)
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))
print('Train ratings:', len(train_df))
print('Test ratings:', len(testset))

Train ratings: 80000
Test ratings: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [9]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**
Для SVD кількість факторів (n_factors) визначає розмір латентного представлення користувачів і фільмів, lr_all — швидкість оновлення параметрів, а reg_all — силу регуляризації, що допомагає уникати перенавчання. Для KNNBasic параметр k задає кількість сусідів, min_k — мінімальну кількість сусідів для прогнозу, name визначає спосіб обчислення схожості, а user_based — чи порівнюються користувачі, чи фільми. Сітка невелика, щоб пошук був обчислювально доступним; GridSearchCV вибере найкращі комбінації за RMSE.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [10]:
# Пошук найкращих параметрів SVD
grid_search_svd = GridSearchCV(
    SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_svd.fit(train_data)
print('SVD GridSearch завершено')

SVD GridSearch завершено


In [14]:
# Пошук найкращих параметрів KNNBasic
grid_search_knn = GridSearchCV(
    KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3, n_jobs=-1
)
grid_search_knn.fit(train_data)
print('KNNBasic GridSearch завершено')

KNNBasic GridSearch завершено


In [12]:
# Результати кросвалідації та найкращі параметри
for name, gs in [('SVD', grid_search_svd), ('KNNBasic', grid_search_knn)]:
    print(f'\\n{name}')
    print('Best CV RMSE:', gs.best_score['rmse'])
    print('Best CV MAE:', gs.best_score['mae'])
    print('Best params (RMSE):', gs.best_params['rmse'])
    print('Best params (MAE):', gs.best_params['mae'])

\nSVD
Best CV RMSE: 0.9422591679971557
Best CV MAE: 0.7433353052834221
Best params (RMSE): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Best params (MAE): {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
\nKNNBasic
Best CV RMSE: 1.028689974772585
Best CV MAE: 0.8159974971282667
Best params (RMSE): {'k': 40, 'min_k': 5, 'sim_options': {'name': 'cosine', 'user_based': True}}
Best params (MAE): {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів
Значення RMSE і MAE та параметри моделей виведені в комірці 5.3. Менші значення метрик означають точніші прогнози. Для остаточного вибору моделі орієнтуємося на CV RMSE, оскільки саме за цією метрикою обирали параметри.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [15]:
# Остаточне навчання моделей на всій train-вибірці
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)
print('Обидві моделі навчено.')

Computing the cosine similarity matrix...
Done computing similarity matrix.
Обидві моделі навчено.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [16]:
# Передбачення на незалежній тестовій вибірці та метрики
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f'SVD: RMSE={svd_rmse:.4f}, MAE={svd_mae:.4f}')
print(f'KNNBasic: RMSE={knn_rmse:.4f}, MAE={knn_mae:.4f}')

SVD: RMSE=0.9192, MAE=0.7234
KNNBasic: RMSE=1.0117, MAE=0.7993


In [17]:
# Порівняльна таблиця тестових результатів
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values('RMSE').reset_index(drop=True)
display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011708,0.799260


**Таблиця тестових результатів:**
Порівняння з CV: RMSE і MAE на тестовій вибірці можна порівняти з показниками кросвалідації, виведеними вище. Невелика різниця є очікуваною, оскільки оцінювання відбувається на різних частинах даних. Значно гірші тестові показники можуть свідчити про складнішу тестову вибірку або нестабільність моделі.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [18]:
# Модель для рекомендацій обирається за найкращим CV RMSE
user_id = '196'
recommendation_model = best_svd if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse'] else best_knn
model_name = 'SVD' if recommendation_model is best_svd else 'KNNBasic'
print('Обрана модель:', model_name)

try:
    user_inner_id = trainset.to_inner_uid(user_id)
    rated_items_inner = trainset.ur[user_inner_id]
    rated_items_raw = {trainset.to_raw_iid(inner_iid) for inner_iid, _ in rated_items_inner}

    all_items_raw = {trainset.to_raw_iid(inner_iid) for inner_iid in trainset.all_items()}
    unrated_items = all_items_raw - rated_items_raw

    recommendations = [
        (iid, recommendation_model.predict(user_id, iid).est)
        for iid in unrated_items
    ]
    top10 = pd.DataFrame(
        sorted(recommendations, key=lambda x: x[1], reverse=True)[:10],
        columns=['item_id', 'predicted_rating']
    )
    display(top10)
except ValueError:
    print(f'Користувача {user_id} немає у навчальній вибірці. Оберіть іншого user_id.')

Обрана модель: SVD


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та
перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки
За результатами кросвалідації обирається модель із нижчим CV RMSE. Найкращі параметри обох моделей і значення CV RMSE/MAE наведено у розділі 5. У розділі 7 наведено незалежні тестові RMSE та MAE, що дають змогу оцінити якість прогнозів на невикористаних під час навчання оцінках. Результати CV та тестування можуть дещо відрізнятися через різний склад вибірок. У розділі 8 сформовано топ-10 фільмів, які обрана модель прогнозує як найцікавіші для користувача. Рекомендаційні алгоритми мають обмеження: для нових користувачів або фільмів без історії оцінок недостатньо даних (cold-start problem), а прогнозовані рейтинги не гарантують фактичних вподобань. Додатково реалізовано простий алгоритм на основі cosine similarity без використання Surprise.


**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [20]:
# Власний простий алгоритм рекомендацій на основі схожості користувачів (cosine)
# Реалізація без Surprise: матриця оцінок, cosine similarity і weighted prediction.
from sklearn.metrics.pairwise import cosine_similarity

ratings_train = train_df[['user_id', 'item_id', 'rating']].copy()
rating_matrix = ratings_train.pivot_table(
    index='user_id', columns='item_id', values='rating'
).fillna(0)

similarity_matrix = cosine_similarity(rating_matrix)
similarity_df = pd.DataFrame(
    similarity_matrix, index=rating_matrix.index, columns=rating_matrix.index
)

custom_user = user_id
if custom_user in rating_matrix.index:
    user_ratings = rating_matrix.loc[custom_user]
    unseen = user_ratings[user_ratings == 0].index
    other_users = rating_matrix.index

    custom_predictions = []
    for movie in unseen:
        movie_ratings = rating_matrix[movie]
        rated_by = movie_ratings[movie_ratings > 0].index
        similarities = similarity_df.loc[custom_user, rated_by]
        values = movie_ratings.loc[rated_by]

        denominator = similarities.abs().sum()
        prediction = (similarities * values).sum() / denominator if denominator > 0 else ratings_train['rating'].mean()
        custom_predictions.append((movie, prediction))

    custom_top10 = pd.DataFrame(
        sorted(custom_predictions, key=lambda x: x[1], reverse=True)[:10],
        columns=['item_id', 'predicted_rating']
    )
    print('Топ-10 рекомендацій власного алгоритму:')
    display(custom_top10)
else:
    print('Користувача немає у train-вибірці для власного алгоритму.')

Топ-10 рекомендацій власного алгоритму:


,item_id,predicted_rating
0,1293,5.0
1,1612,5.0
2,1189,5.0
3,1201,5.0
4,1306,5.0
5,1467,5.0
6,1500,5.0
7,1629,5.0
8,1642,5.0
9,1653,5.0
